# Phase 9 — Estimation, treated authors

Dynamic event studies on the treated panel, before matched controls enter.
Coefficients are measured against the author's own reference period, so they
compare arms to each other and lead authors to middle authors on the same paper.
Phase 12 supplies the absolute version against matched controls.

**Inputs:** `data/interim/phase08_panel_extended.csv`,
`data/interim/phase08_panel_citations.csv`

**Output:** `data/results/phase09_results.csv`

Two event times (-2, -1) are omitted rather than one, and year fixed effects are
off here, because among treated authors alone calendar year and event time are
near-collinear; Phase 12 turns them on once controls break that collinearity.

In [1]:
import math
import os

import numpy as np
import pandas as pd

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

PANEL_EXT = "data/interim/phase08_panel_extended.csv"
PANEL_CIT = "data/interim/phase08_panel_citations.csv"
OUT_RESULTS = "data/results/phase09_results.csv"

ARMS = ["AUTHOR_MISCONDUCT", "HONEST_ERROR", "EDITORIAL_COMPROMISE"]

REF_EVENTS = (-2, -1)

ANALYSIS_PRE, ANALYSIS_POST = 6, 6
CIT_ANALYSIS_PRE, CIT_ANALYSIS_POST = 3, 3

YEAR_FE = False
CLUSTER_VAR = "rw_journal"

PLACEBO_OFFSET = -6

MIN_MEANINGFUL_EXIT = 0.02
MIN_MEANINGFUL_PROP = 0.05

MIN_AUTHORS_PER_SPLIT = 100

pd.set_option("display.width", 220)
os.makedirs("data/results", exist_ok=True)

print(f"reference period   {REF_EVENTS}")
print(f"year fixed effects {YEAR_FE}")
print(f"clustering on      {CLUSTER_VAR}")
print(f"placebo offset     {PLACEBO_OFFSET}")

reference period   (-2, -1)
year fixed effects False
clustering on      rw_journal
placebo offset     -6


## Estimation engine

In [2]:
def _norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def absorb_fe(df, cols, groups, tol=1e-10, max_iter=100):
    """Alternating projections run to convergence."""
    out = df[cols].astype(float).copy()
    for _ in range(max_iter):
        prev = out.values.copy()
        for g in groups:
            gg = df[g]
            for c in cols:
                out[c] = out[c] - out[c].groupby(gg).transform("mean")
        if np.abs(out.values - prev).max() < tol:
            break
    return out


def ols_cluster(y, X, cluster, names):
    """OLS with cluster-robust standard errors."""
    y = np.asarray(y, float)
    X = np.asarray(X, float)
    n, k = X.shape
    XtX_inv = np.linalg.pinv(X.T @ X)
    beta = XtX_inv @ (X.T @ y)
    resid = y - X @ beta

    cl = pd.Series(cluster).astype(str).values
    order = np.argsort(cl)
    cl_s, X_s, r_s = cl[order], X[order], resid[order]
    bounds = np.flatnonzero(np.r_[True, cl_s[1:] != cl_s[:-1], True])
    meat = np.zeros((k, k))
    for a, b in zip(bounds[:-1], bounds[1:]):
        s = X_s[a:b].T @ r_s[a:b]
        meat += np.outer(s, s)

    G = len(bounds) - 1
    dof = (G / max(G - 1, 1)) * ((n - 1) / max(n - k, 1))
    V = dof * (XtX_inv @ meat @ XtX_inv)
    se = np.sqrt(np.clip(np.diag(V), 0, None))
    with np.errstate(divide="ignore", invalid="ignore"):
        t = np.where(se > 0, beta / se, np.nan)
    p = np.array([2 * (1 - _norm_cdf(abs(ti))) if np.isfinite(ti) else np.nan
                  for ti in t])

    res = pd.DataFrame({"term": names, "coef": beta, "se": se, "t": t, "p": p,
                        "ci_lo": beta - 1.96 * se, "ci_hi": beta + 1.96 * se})
    res.attrs["n_clusters"] = G
    return res


def check_design(X):
    """Flag a rank-deficient or ill-conditioned design; pinv would otherwise
    return a minimum-norm solution for a singular system."""
    if np.linalg.matrix_rank(X) < X.shape[1]:
        return False, "rank-deficient", np.inf
    c = float(np.linalg.cond(X.T @ X))
    return (c <= 1e10), ("" if c <= 1e10 else "ill-conditioned"), c

In [3]:
def event_study(panel, outcome, group_col, groups, spec,
                pre=None, post=None, year_fe=YEAR_FE):
    """One coefficient per group per event time, against the reference period."""
    pre = ANALYSIS_PRE if pre is None else pre
    post = ANALYSIS_POST if post is None else post

    d = panel.dropna(subset=[outcome, CLUSTER_VAR, group_col]).copy()
    d = d[(d.event_time >= -pre) & (d.event_time <= post)]
    d = d[d[group_col].astype(str).isin([str(g) for g in groups])]
    if d.empty:
        return None

    ks = [k for k in sorted(d.event_time.unique()) if k not in REF_EVENTS]
    ev = pd.DataFrame({f"k{k:+d}": (d.event_time == k).astype(float)
                       for k in ks}, index=d.index)

    parts, names = [], []
    present = [g for g in groups if str(g) in set(d[group_col].astype(str))]
    for g in present:
        m = (d[group_col].astype(str) == str(g)).astype(float).values[:, None]
        parts.append(ev.values * m)
        names += [f"{c}:{g}" for c in ev.columns]
    X = np.hstack(parts)
    y = d[outcome].values.astype(float)

    fe = ["author_id"] + (["year"] if year_fe else [])
    tmp = pd.DataFrame(X, columns=[f"c{i}" for i in range(X.shape[1])],
                       index=d.index)
    tmp[outcome] = y
    for g in fe:
        tmp[g] = d[g].values
    ab = absorb_fe(tmp, [f"c{i}" for i in range(X.shape[1])] + [outcome], fe)
    X = ab[[f"c{i}" for i in range(X.shape[1])]].values
    y = ab[outcome].values

    ok, why, cond = check_design(X)
    print(f"  {spec}: {len(d):,} rows | {d.author_id.nunique():,} authors "
          f"| cond(X'X) {cond:.2e}")
    if not ok:
        print(f"    [!] design {why}; not estimated")
        return None

    res = ols_cluster(y, X, d[CLUSTER_VAR].values, names)
    parsed = res.term.str.extract(r"^k([+-]?\d+):(.+)$")
    res["event_time"] = pd.to_numeric(parsed[0], errors="coerce")
    res["group"] = parsed[1]
    res["outcome"] = outcome
    res["spec"] = spec
    res["n_obs"] = len(d)
    res["n_authors"] = d.author_id.nunique()
    res["n_clusters"] = res.attrs["n_clusters"]
    res["cond"] = cond
    return res.dropna(subset=["event_time"]).sort_values(["group", "event_time"])

## Reporting

In [4]:
def stars(p):
    return ("***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "")


def report(res, title, threshold=None, horizons=(1, 3, 6)):
    if res is None:
        return
    d = res.copy()
    d["sig"] = d.p.apply(stars)
    if threshold is not None:
        d["meaningful"] = d.coef.abs() >= threshold
    sel = d[d.event_time.isin(horizons)]
    cols = ["group", "event_time", "coef", "se", "p", "sig", "ci_lo", "ci_hi"]
    if threshold is not None:
        cols.append("meaningful")
    print(f"\n{title}")
    print(sel[cols].round(4).to_string(index=False))
    at_last = d[d.event_time == max(horizons)]
    if len(at_last) > 1:
        spread = float(at_last.coef.max() - at_last.coef.min())
        print(f"  spread across groups at +{max(horizons)}: {spread:.4f}")


def parallel_trends(res, title):
    if res is None:
        return
    rows = []
    for g, sub in res[res.event_time < 0].groupby("group"):
        sub = sub.sort_values("event_time")
        slope = (float(np.polyfit(sub.event_time, sub.coef, 1)[0])
                 if len(sub) >= 2 else np.nan)
        rows.append({"group": g, "n_coef": len(sub),
                     "n_sig": int((sub.p < 0.05).sum()),
                     "mean_abs": round(float(sub.coef.abs().mean()), 4),
                     "slope_per_year": round(slope, 4)})
    if not rows:
        return
    t = pd.DataFrame(rows)
    print(f"\npre-period slopes — {title}")
    print(t.to_string(index=False))
    if t.slope_per_year.notna().sum() > 1:
        spread = float(t.slope_per_year.max() - t.slope_per_year.min())
        print(f"  spread of slopes across groups: {spread:.4f}")

## Load

In [5]:
ext = pd.read_csv(PANEL_EXT, low_memory=False)
ext = ext[ext.balanced & ext.in_primary]
print(f"extended panel  {len(ext):,} rows, {ext.author_id.nunique():,} authors")
print(ext.groupby("author_id").arm.first().value_counts().to_string())

cit = None
if os.path.isfile(PANEL_CIT):
    cit = pd.read_csv(PANEL_CIT, low_memory=False)
    cit = cit[cit.balanced & cit.in_primary]
    print(f"\ncitation panel  {len(cit):,} rows, "
          f"{cit.author_id.nunique():,} authors")

results = []

extended panel  269,710 rows, 19,265 authors
arm
AUTHOR_MISCONDUCT       9617
HONEST_ERROR            4894
EDITORIAL_COMPROMISE    2783
UNCONFIRMED_CONCERNS    1457
ETHICS_VIOLATION         327
UNCLASSIFIED             187

citation panel  245,768 rows, 25,486 authors


## Exit and output, by arm

In [6]:
for outcome, thr, label in [("active", MIN_MEANINGFUL_EXIT, "exit"),
                            ("publications", None, "publications")]:
    if outcome not in ext.columns:
        continue
    r = event_study(ext, outcome, "arm", ARMS, f"{label} ~ arm")
    if r is None:
        continue
    results.append(r)
    parallel_trends(r, f"{label} ~ arm")
    report(r, f"{label.upper()} ~ arm", thr)

  exit ~ arm: 224,822 rows | 17,294 authors | cond(X'X) 2.25e+01

pre-period slopes — exit ~ arm
               group  n_coef  n_sig  mean_abs  slope_per_year
   AUTHOR_MISCONDUCT       4      4    0.0622          0.0315
EDITORIAL_COMPROMISE       4      4    0.1126          0.0381
        HONEST_ERROR       4      4    0.0842          0.0353
  spread of slopes across groups: 0.0066

EXIT ~ arm
               group  event_time    coef     se      p sig   ci_lo   ci_hi  meaningful
   AUTHOR_MISCONDUCT           1 -0.0575 0.0051 0.0000 *** -0.0674 -0.0476        True
   AUTHOR_MISCONDUCT           3 -0.0802 0.0051 0.0000 *** -0.0901 -0.0702        True
   AUTHOR_MISCONDUCT           6 -0.0984 0.0051 0.0000 *** -0.1084 -0.0884        True
EDITORIAL_COMPROMISE           1 -0.0413 0.0130 0.0014  ** -0.0667 -0.0159        True
EDITORIAL_COMPROMISE           3 -0.0589 0.0113 0.0000 *** -0.0811 -0.0368        True
EDITORIAL_COMPROMISE           6 -0.0841 0.0111 0.0000 *** -0.1059 -0.0623      

## Byline position, within the retracted paper

First and last authors against middle authors on the same paper, holding field,
venue, era, and the notice constant. Career stage is not held constant, which
the placebo does not separate.

In [7]:
pos = ext[ext.first_position.astype(str).isin(["first", "middle", "last"])].copy()
pos["is_lead"] = pos.first_position.astype(str).isin(["first", "last"])

for outcome, thr, label in [("publications", None, "publications"),
                            ("active", MIN_MEANINGFUL_EXIT, "active")]:
    if outcome not in pos.columns:
        continue
    r = event_study(pos, outcome, "is_lead", [True, False],
                    f"{label} ~ position")
    if r is None:
        continue
    results.append(r)
    parallel_trends(r, f"{label} ~ position")
    report(r, f"{label.upper()} ~ position", thr, horizons=(1, 2, 3))

    lead = r[r.group == "True"].set_index("event_time").coef
    mid = r[r.group == "False"].set_index("event_time").coef
    ks = sorted(set(lead.index) & set(mid.index) & {1, 2, 3})
    if ks:
        print("  middle minus lead")
        for k in ks:
            print(f"    +{k}:  {mid[k] - lead[k]:+.4f}")

  publications ~ position: 250,445 rows | 19,265 authors | cond(X'X) 1.12e+01

pre-period slopes — publications ~ position
group  n_coef  n_sig  mean_abs  slope_per_year
False       4      4    1.2312          0.3750
 True       4      4    1.1418          0.3312
  spread of slopes across groups: 0.0438

PUBLICATIONS ~ position
group  event_time   coef     se      p sig   ci_lo  ci_hi
False           1 0.3584 0.0807 0.0000 ***  0.2003 0.5165
False           2 0.4659 0.0784 0.0000 ***  0.3123 0.6195
False           3 0.5306 0.0937 0.0000 ***  0.3469 0.7142
 True           1 0.1399 0.0870 0.1077     -0.0306 0.3104
 True           2 0.2481 0.0979 0.0113   *  0.0561 0.4400
 True           3 0.3780 0.1170 0.0012  **  0.1487 0.6072
  spread across groups at +3: 0.1526
  middle minus lead
    +1:  +0.2184
    +2:  +0.2179
    +3:  +0.1526
  active ~ position: 250,445 rows | 19,265 authors | cond(X'X) 1.12e+01

pre-period slopes — active ~ position
group  n_coef  n_sig  mean_abs  slope_per_yea

## Placebo

The same comparison at a fake event four years earlier, on the same authors. A
gap at that date is a pre-existing divergence rather than a response to the
retraction.

In [8]:
plac = pos.copy()
plac["event_time"] = plac.event_time - PLACEBO_OFFSET
plac = plac[(plac.event_time >= -3) & (plac.event_time <= 3)]

for outcome, label in [("publications", "publications"),
                       ("citations", "citations")]:
    src = plac if outcome in plac.columns else None
    if src is None and cit is not None and outcome in cit.columns:
        c = cit[cit.first_position.astype(str)
                   .isin(["first", "middle", "last"])].copy()
        c["is_lead"] = c.first_position.astype(str).isin(["first", "last"])
        c["event_time"] = c.event_time - PLACEBO_OFFSET
        src = c[(c.event_time >= -3) & (c.event_time <= 3)]
    if src is None:
        continue

    r = event_study(src, outcome, "is_lead", [True, False],
                    f"PLACEBO {label} ~ position", pre=3, post=3)
    if r is None:
        continue
    results.append(r)

    lead = r[r.group == "True"].set_index("event_time").coef
    mid = r[r.group == "False"].set_index("event_time").coef
    ks = sorted(set(lead.index) & set(mid.index) & {1, 2, 3})
    if ks:
        print(f"  {label}, middle minus lead at the fake event")
        for k in ks:
            print(f"    +{k}:  {mid[k] - lead[k]:+.4f}")

  PLACEBO publications ~ position: 96,325 rows | 19,265 authors | cond(X'X) 8.61e+00
  publications, middle minus lead at the fake event
    +1:  +0.0621
    +2:  -0.0110
    +3:  +0.0488
  PLACEBO citations ~ position: 92,852 rows | 25,486 authors | cond(X'X) 1.80e+01
  citations, middle minus lead at the fake event
    +1:  +8.7232
    +2:  +11.9078
    +3:  +16.9260


## Citations to other work

The retracted paper is excluded, so this measures whether an author's remaining
work is cited differently after the retraction. Annual citations rise for any
active researcher, so only the comparison against the placebo carries
information.

In [9]:
if cit is not None and "citations" in cit.columns:
    c = cit[cit.first_position.astype(str)
               .isin(["first", "middle", "last"])].copy()
    c["is_lead"] = c.first_position.astype(str).isin(["first", "last"])

    r = event_study(c, "citations", "is_lead", [True, False],
                    "citations ~ position",
                    pre=CIT_ANALYSIS_PRE, post=CIT_ANALYSIS_POST)
    if r is not None:
        results.append(r)
        parallel_trends(r, "citations ~ position")
        report(r, "CITATIONS ~ position", horizons=(1, 2, 3))

    r2 = event_study(c, "citations", "arm", ARMS, "citations ~ arm",
                     pre=CIT_ANALYSIS_PRE, post=CIT_ANALYSIS_POST)
    if r2 is not None:
        results.append(r2)
        parallel_trends(r2, "citations ~ arm")
        report(r2, "CITATIONS ~ arm", horizons=(1, 2, 3))
else:
    print("citation panel unavailable")

  citations ~ position: 178,402 rows | 25,486 authors | cond(X'X) 6.05e+00

pre-period slopes — citations ~ position
group  n_coef  n_sig  mean_abs  slope_per_year
False       1      1   35.3282             NaN
 True       1      1   30.0469             NaN

CITATIONS ~ position
group  event_time     coef      se   p sig    ci_lo    ci_hi
False           1 110.4397  8.0414 0.0 ***  94.6786 126.2007
False           2 149.9360 10.2567 0.0 *** 129.8329 170.0391
False           3 177.1241  9.3984 0.0 *** 158.7032 195.5450
 True           1  86.3962  4.4159 0.0 ***  77.7412  95.0513
 True           2 118.3291  5.3059 0.0 *** 107.9295 128.7286
 True           3 150.2779  6.4314 0.0 *** 137.6723 162.8834
  spread across groups at +3: 26.8462
  citations ~ arm: 158,431 rows | 22,633 authors | cond(X'X) 1.39e+01

pre-period slopes — citations ~ arm
               group  n_coef  n_sig  mean_abs  slope_per_year
   AUTHOR_MISCONDUCT       1      1   25.8063             NaN
EDITORIAL_COMPROMISE    

## Heterogeneity

Splits declared in advance. A split below the minimum author count is skipped
rather than reported with intervals wide enough to accommodate anything.

In [10]:
for var in ["career_band", "income_group", "subject_group", "lag_band"]:
    if var not in ext.columns or ext[var].isna().all():
        continue
    print(f"\nexit by arm, split on {var}")
    for level, sub in ext.groupby(var, observed=True):
        n = sub.author_id.nunique()
        if n < MIN_AUTHORS_PER_SPLIT:
            print(f"  {var}={level}: {n:,} authors, below "
                  f"{MIN_AUTHORS_PER_SPLIT}; skipped")
            continue
        r = event_study(sub, "active", "arm", ARMS,
                        f"active ~ arm | {var}={level}")
        if r is None:
            continue
        r[var] = level
        results.append(r)
        at6 = r[r.event_time == ANALYSIS_POST]
        vals = ", ".join(f"{g[:4]} {c:+.4f}"
                         for g, c in zip(at6.group, at6.coef))
        print(f"  {var}={level}: {n:,} authors | at +{ANALYSIS_POST}: {vals}")


exit by arm, split on career_band
  active ~ arm | career_band=early (<5y): 13,962 rows | 1,074 authors | cond(X'X) 1.89e+01
  career_band=early (<5y): 1,184 authors | at +6: AUTH -0.2056, EDIT -0.1609, HONE -0.2273
  active ~ arm | career_band=mid (5-10y): 36,530 rows | 2,810 authors | cond(X'X) 2.31e+01
  career_band=mid (5-10y): 3,124 authors | at +6: AUTH -0.1293, EDIT -0.0866, HONE -0.1059
  active ~ arm | career_band=senior (10-20y): 72,605 rows | 5,585 authors | cond(X'X) 2.07e+01
  career_band=senior (10-20y): 6,166 authors | at +6: AUTH -0.0758, EDIT -0.0685, HONE -0.0789
  active ~ arm | career_band=veteran (20y+): 101,725 rows | 7,825 authors | cond(X'X) 2.43e+01
  career_band=veteran (20y+): 8,791 authors | at +6: AUTH -0.0882, EDIT -0.0827, HONE -0.0918

exit by arm, split on subject_group
  active ~ arm | subject_group=B/T: 18,031 rows | 1,387 authors | cond(X'X) 2.70e+01
  subject_group=B/T: 1,508 authors | at +6: AUTH -0.1101, EDIT -0.0331, HONE -0.0740
  active ~ arm 

## Write

In [11]:
if not results:
    print("nothing estimated")
else:
    out = pd.concat(results, ignore_index=True)
    for c in ["career_band", "income_group", "subject_group", "lag_band"]:
        if c not in out.columns:
            out[c] = np.nan
    out["meaningful"] = np.where(out.outcome == "active",
                                 out.coef.abs() >= MIN_MEANINGFUL_EXIT,
                                 np.nan)
    out.to_csv(OUT_RESULTS, index=False)
    print(f"{OUT_RESULTS}: {len(out):,} coefficient rows")
    print(out.spec.value_counts().to_string())

data/results/phase09_results.csv: 580 coefficient rows
spec
exit ~ arm                                    33
publications ~ arm                            33
active ~ arm | career_band=senior (10-20y)    33
active ~ arm | career_band=veteran (20y+)     33
active ~ arm | career_band=early (<5y)        33
active ~ arm | subject_group=SOC              33
active ~ arm | lag_band=fast (<1y)            33
active ~ arm | lag_band=medium (1-3y)         33
active ~ arm | career_band=mid (5-10y)        33
active ~ arm | subject_group=B/T              33
active ~ arm | subject_group=HSC              33
active ~ arm | subject_group=ENV              33
active ~ arm | subject_group=BLS              33
active ~ arm | lag_band=slow (3y+)            33
active ~ arm | subject_group=PHY              33
publications ~ position                       22
active ~ position                             22
citations ~ arm                               15
citations ~ position                          10
PLACEBO p